# CHICAGO CRIME ANALYSIS — NOTEBOOK FINAL CONSOLIDÉ

## Découverte de connaissances dans une base de données multidimensionnelle

**Dataset :** Chicago Crime Dataset — secteur de Bridgeport  
**Responsable du groupe :** Angelikia Kavuansiko  
**Membres et répartition du travail :**

| Membre | Contribution principale |
|---|---|
| Angelikia Kavuansiko | Exploration des données et requête de regroupement |
| Ekta | Pattern mining avec Apriori et règles d'association |
| Léora | Analyse temporelle et prévision |
| Chrisa | Analyse spatiale et clustering |
| Flavie | Dashboard, présentation et intégration finale |

**Lien du dépôt GitHub :** `À COMPLÉTER AVANT LE DÉPÔT`  

> **Attention méthodologique :** le fichier fourni contient des incidents localisés dans le secteur de **Bridgeport**. Les résultats décrivent donc ce périmètre et ne doivent pas être généralisés sans précaution à l'ensemble de Chicago.

## 1. Objectif et démarche KDD

La problématique du projet est la suivante :

> **Comment les crimes enregistrés dans le secteur de Bridgeport se répartissent-ils dans le temps et dans l'espace, et quels motifs récurrents peut-on identifier afin de mieux comprendre leur évolution ?**

Le notebook suit les principales étapes du processus **Knowledge Discovery in Databases (KDD)** :

1. compréhension et chargement des données ;
2. nettoyage et transformation ;
3. construction d'indicateurs descriptifs ;
4. recherche de motifs fréquents ;
5. analyse temporelle et prévision ;
6. analyse spatiale et clustering ;
7. intégration des résultats dans un dashboard.

Toutes les méthodes importantes sont regroupées dans des fonctions. Chaque fonction précise son **nom**, ses **entrées** et ses **sorties**, conformément aux consignes du projet.

## 2. Déclaration relative à l'utilisation d'un LLM

Une aide par modèle de langage a été utilisée pour consolider, commenter et sécuriser le code provenant des quatre notebooks de travail.

**Prompt utilisé :**

> « À partir des notebooks `exploration.ipynb`, `pattern_mining.ipynb`, `temporal_analysis.ipynb` et `spatial_analysis_v1.ipynb`, créer un notebook final consolidé. Organiser chaque méthode dans une fonction documentée avec son nom, ses entrées et ses sorties. Ajouter une explication en langage courant pour chaque indicateur, conserver les analyses originales, corriger les incohérences méthodologiques, ajouter un bloc principal appelant toutes les fonctions et prévoir le lancement du dashboard Dash. »

Le code reste relu, testé et interprété par l'équipe. L'utilisation du LLM ne remplace pas la compréhension des méthodes employées.

## 3. Importation des bibliothèques et configuration

In [1]:
# Bibliothèques standards
from pathlib import Path
import subprocess
import sys
import warnings

# Manipulation et calcul
import numpy as np
import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

# Pattern mining
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Analyse spatiale
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans, OPTICS

# Prévision : Prophet est privilégié, mais une méthode de secours est prévue
# afin que le notebook reste exécutable si Prophet n'est pas disponible.
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception as exc:
    Prophet = None
    PROPHET_AVAILABLE = False
    print(f"Prophet indisponible : {exc}")

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

ModuleNotFoundError: No module named 'mlxtend'

### Dépendances nécessaires

Depuis la racine du dépôt, les bibliothèques du dashboard et du notebook peuvent être installées avec :

```bash
python -m pip install -r dashboard/requirements.txt
```

Pour exécuter le notebook dans Jupyter :

```bash
jupyter notebook notebooks/Kavuansiko_notebook.ipynb
```

## 4. Fonctions communes : localisation du projet et chargement des données

In [2]:
# Fonction : resolve_project_root
# Input  : aucun paramètre obligatoire ; la fonction inspecte le dossier courant.
# Output : chemin Path correspondant à la racine du projet.
def resolve_project_root() -> Path:
    """Retrouver automatiquement la racine du dépôt.

    La fonction permet d'exécuter le notebook depuis la racine du projet,
    depuis le dossier notebooks/ ou depuis un environnement Jupyter différent.
    """
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

    for candidate in candidates:
        if (candidate / "data" / "chicago_crime.csv").exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Impossible de localiser data/chicago_crime.csv. "
        "Placez ce notebook dans le dossier notebooks/ du projet."
    )


# Fonction : load_data
# Input  : chemin vers le fichier CSV.
# Output : DataFrame pandas avec dates et coordonnées converties.
def load_data(file_path: str | Path) -> pd.DataFrame:
    """Charger le fichier CSV et convertir les variables essentielles."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Fichier de données introuvable : {path}")

    df = pd.read_csv(path)

    # Les dates sont au format américain mois/jour/année avec heure AM/PM.
    df["Date"] = pd.to_datetime(
        df["Date"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce",
    )

    # Le fichier utilise une virgule comme séparateur décimal pour les coordonnées.
    for column in ["Latitude", "Longitude"]:
        df[column] = pd.to_numeric(
            df[column].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

    # Variables temporelles réutilisées dans plusieurs analyses.
    df["Year"] = df["Date"].dt.year.astype("Int64")
    df["Month"] = df["Date"].dt.month.astype("Int64")
    df["Hour"] = df["Date"].dt.hour.astype("Int64")
    df["YearMonth"] = df["Date"].dt.to_period("M").dt.to_timestamp()

    return df

## 5. Requête 1 — Exploration des données et regroupements

### 5.1 Signification du dataset

Chaque ligne représente un incident criminel déclaré. Les trois dimensions du dataset sont :

- **temporelle** : date, heure et année de l'incident ;
- **spatiale** : bloc, beat, ward, coordonnées X/Y, latitude et longitude ;
- **analytique** : type de crime, description, lieu, arrestation et caractère domestique.

L'exploration vérifie la taille du fichier, le type des variables, les données manquantes et la plage de valeurs de chaque colonne. Cette étape conditionne la fiabilité des analyses suivantes.

In [3]:
# Fonction : build_variable_summary
# Input  : DataFrame complet.
# Output : tableau récapitulatif du type, des valeurs manquantes et de la plage de chaque variable.
def build_variable_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Construire un résumé humainement lisible pour chaque colonne."""
    rows = []

    for column in df.columns:
        series = df[column]
        non_null = series.dropna()
        missing_count = int(series.isna().sum())
        missing_rate = round(missing_count / len(df) * 100, 2)
        unique_count = int(non_null.nunique())

        minimum = None
        maximum = None

        # Pour les nombres et les dates, la plage correspond au minimum et au maximum.
        if pd.api.types.is_numeric_dtype(series) or pd.api.types.is_datetime64_any_dtype(series):
            if not non_null.empty:
                minimum = non_null.min()
                maximum = non_null.max()
        else:
            # Pour une variable catégorielle, on indique plutôt la modalité la plus fréquente.
            if not non_null.empty:
                minimum = f"Modalité fréquente : {non_null.mode().iloc[0]}"
                maximum = f"{unique_count} modalités distinctes"

        rows.append(
            {
                "Variable": column,
                "Type": str(series.dtype),
                "Valeurs manquantes": missing_count,
                "Taux manquant (%)": missing_rate,
                "Valeurs distinctes": unique_count,
                "Minimum / information 1": minimum,
                "Maximum / information 2": maximum,
            }
        )

    return pd.DataFrame(rows)


# Fonction : compute_dataset_overview
# Input  : DataFrame complet.
# Output : dictionnaire contenant les indicateurs généraux du dataset.
def compute_dataset_overview(df: pd.DataFrame) -> dict:
    """Calculer les principaux indicateurs d'exploration."""
    return {
        "Nombre de lignes": len(df),
        "Nombre de colonnes": df.shape[1],
        "Date minimale": df["Date"].min(),
        "Date maximale": df["Date"].max(),
        "Types de crimes distincts": df["Primary Type"].nunique(),
        "Taux global d'arrestation (%)": round(df["Arrest"].mean() * 100, 2),
        "Lignes sans coordonnées complètes": int(
            df[["Latitude", "Longitude"]].isna().any(axis=1).sum()
        ),
    }


# Fonction : compute_top_crimes
# Input  : DataFrame et nombre maximum de catégories.
# Output : tableau des types de crimes les plus fréquents.
def compute_top_crimes(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """Compter les incidents par type de crime et retenir les plus fréquents."""
    return (
        df.groupby("Primary Type", observed=True)
        .size()
        .reset_index(name="Nombre_incidents")
        .sort_values("Nombre_incidents", ascending=False)
        .head(top_n)
    )


# Fonction : plot_top_crimes
# Input  : tableau produit par compute_top_crimes.
# Output : figure Plotly en barres horizontales.
def plot_top_crimes(top_crimes: pd.DataFrame) -> go.Figure:
    """Visualiser le classement des crimes les plus fréquents."""
    ordered = top_crimes.sort_values("Nombre_incidents")
    fig = px.bar(
        ordered,
        x="Nombre_incidents",
        y="Primary Type",
        orientation="h",
        text="Nombre_incidents",
        title="Top 10 des types de crimes enregistrés à Bridgeport",
        labels={"Primary Type": "Type de crime", "Nombre_incidents": "Nombre d'incidents"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=500)
    return fig


# Fonction : compute_arrest_rates
# Input  : DataFrame et nombre de types de crimes étudiés.
# Output : tableau du taux d'arrestation pour les crimes les plus représentés.
def compute_arrest_rates(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """Calculer les arrestations et leur taux par type de crime."""
    grouped = (
        df.groupby("Primary Type", observed=True)
        .agg(
            Nombre_incidents=("Arrest", "size"),
            Nombre_arrestations=("Arrest", "sum"),
        )
        .sort_values("Nombre_incidents", ascending=False)
        .head(top_n)
        .reset_index()
    )
    grouped["Taux_arrestation_pct"] = (
        grouped["Nombre_arrestations"] / grouped["Nombre_incidents"] * 100
    ).round(2)
    return grouped.sort_values("Taux_arrestation_pct", ascending=False)


# Fonction : plot_arrest_rates
# Input  : tableau des taux d'arrestation.
# Output : figure Plotly en barres.
def plot_arrest_rates(arrest_rates: pd.DataFrame) -> go.Figure:
    """Visualiser le taux d'arrestation des crimes les plus fréquents."""
    ordered = arrest_rates.sort_values("Taux_arrestation_pct")
    fig = px.bar(
        ordered,
        x="Taux_arrestation_pct",
        y="Primary Type",
        orientation="h",
        text="Taux_arrestation_pct",
        hover_data=["Nombre_incidents", "Nombre_arrestations"],
        title="Taux d'arrestation parmi les dix crimes les plus fréquents",
        labels={
            "Primary Type": "Type de crime",
            "Taux_arrestation_pct": "Taux d'arrestation (%)",
        },
    )
    fig.update_traces(texttemplate="%{text:.1f} %", textposition="outside")
    fig.update_xaxes(range=[0, 100])
    fig.update_layout(height=500)
    return fig

### 5.2 Calcul et interprétation des indicateurs

#### Nombre d'incidents par type de crime

Le nombre d'incidents est obtenu par un regroupement sur `Primary Type`, puis par comptage des lignes. Il mesure la fréquence brute de chaque catégorie dans le fichier.

**Interprétation :** une barre plus longue signifie qu'un type de crime est davantage représenté dans le dataset. Cet indicateur décrit la fréquence enregistrée, mais ne mesure ni la gravité ni le risque individuel.

#### Taux d'arrestation

Pour chaque type de crime :

\[
\text{Taux d'arrestation} = \frac{\text{nombre d'incidents avec Arrest = True}}{\text{nombre total d'incidents du type}} \times 100
\]

Le graphique porte sur les dix types de crimes les plus fréquents afin d'éviter qu'une catégorie très rare paraisse artificiellement performante avec, par exemple, une arrestation sur un seul incident.

**Interprétation :** un taux élevé signifie qu'une part importante des incidents enregistrés pour cette catégorie a conduit à une arrestation. Il ne permet pas, à lui seul, de conclure sur l'efficacité globale des services de police.

## 6. Requête 2 — Pattern mining avec Apriori

### 6.1 Principe

L'algorithme **Apriori** recherche des associations récurrentes entre plusieurs caractéristiques d'un incident. Les variables doivent d'abord être converties en catégories :

| Variable | Transformation retenue |
|---|---|
| Heure | Matin, après-midi, soir ou nuit |
| Lieu | Cinq lieux les plus fréquents, puis catégorie `AUTRE` |
| Arrestation | `Arrestation_OUI` ou `Arrestation_NON` |
| Caractère domestique | `Domestique_OUI` ou `Domestique_NON` |
| Type de crime | Catégorie d'origine |

Une transaction correspond à une ligne du dataset et contient cinq items.

In [4]:
# Fonction : discretise_for_apriori
# Input  : DataFrame nettoyé.
# Output : DataFrame composé uniquement de variables catégorielles utilisables par Apriori.
def discretise_for_apriori(df: pd.DataFrame) -> pd.DataFrame:
    """Transformer les variables en items catégoriels."""
    result = df.copy()

    def time_slot(hour) -> str:
        if pd.isna(hour):
            return "Heure_inconnue"
        hour = int(hour)
        if 6 <= hour < 12:
            return "Matin"
        if 12 <= hour < 18:
            return "Après-midi"
        if 18 <= hour < 23:
            return "Soir"
        return "Nuit"

    top_locations = result["Location Description"].value_counts().nlargest(5).index

    result["Heure"] = result["Hour"].apply(time_slot)
    result["Lieu"] = result["Location Description"].where(
        result["Location Description"].isin(top_locations),
        "AUTRE",
    )
    result["Arrestation"] = np.where(
        result["Arrest"], "Arrestation_OUI", "Arrestation_NON"
    )
    result["Domestique"] = np.where(
        result["Domestic"], "Domestique_OUI", "Domestique_NON"
    )

    return result[["Primary Type", "Heure", "Lieu", "Arrestation", "Domestique"]]


# Fonction : encode_transactions
# Input  : DataFrame discrétisé.
# Output : matrice booléenne où chaque colonne correspond à un item.
def encode_transactions(discrete_df: pd.DataFrame) -> pd.DataFrame:
    """Encoder les transactions avec TransactionEncoder."""
    transactions = discrete_df.apply(lambda row: list(row.values), axis=1).tolist()
    encoder = TransactionEncoder()
    encoded_array = encoder.fit(transactions).transform(transactions)
    return pd.DataFrame(encoded_array, columns=encoder.columns_)


# Fonction : extract_frequent_patterns
# Input  : matrice encodée, support minimal et lift minimal.
# Output : deux DataFrames : itemsets fréquents et règles d'association.
def extract_frequent_patterns(
    encoded_df: pd.DataFrame,
    min_support: float = 0.10,
    min_lift: float = 1.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Extraire les itemsets fréquents et les règles d'association."""
    itemsets = apriori(
        encoded_df,
        min_support=min_support,
        use_colnames=True,
    ).sort_values("support", ascending=False)

    # Une règle n'est générée que si suffisamment d'itemsets existent.
    if itemsets.empty:
        rules = pd.DataFrame()
    else:
        rules = association_rules(
            itemsets,
            metric="lift",
            min_threshold=min_lift,
        ).sort_values("lift", ascending=False)

    return itemsets, rules


# Fonction : compute_support_sensitivity
# Input  : matrice encodée et liste de supports minimaux.
# Output : tableau du nombre d'itemsets obtenu pour chaque support.
def compute_support_sensitivity(
    encoded_df: pd.DataFrame,
    supports: list[float] | None = None,
) -> pd.DataFrame:
    """Mesurer l'effet du support minimal sur le nombre de motifs détectés."""
    if supports is None:
        supports = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

    rows = []
    for support in supports:
        count = len(apriori(encoded_df, min_support=support, use_colnames=True))
        rows.append({"Support_minimal": support, "Nombre_itemsets": count})

    return pd.DataFrame(rows)


# Fonction : plot_support_sensitivity
# Input  : tableau produit par compute_support_sensitivity.
# Output : figure Plotly représentant la sensibilité au support.
def plot_support_sensitivity(sensitivity: pd.DataFrame) -> go.Figure:
    """Visualiser l'effet du seuil de support."""
    fig = px.line(
        sensitivity,
        x="Support_minimal",
        y="Nombre_itemsets",
        markers=True,
        title="Impact du support minimal sur le nombre d'itemsets fréquents",
        labels={
            "Support_minimal": "Support minimal",
            "Nombre_itemsets": "Nombre d'itemsets fréquents",
        },
    )
    fig.add_vline(x=0.10, line_dash="dash", annotation_text="Seuil retenu : 0,10")
    return fig


# Fonction : format_rules_table
# Input  : DataFrame brut des règles et nombre de règles à conserver.
# Output : tableau lisible des principales règles.
def format_rules_table(rules: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    """Convertir les frozensets Apriori en texte lisible."""
    if rules.empty:
        return pd.DataFrame(
            columns=["Antécédent", "Conséquent", "Support", "Confiance", "Lift"]
        )

    result = rules.head(top_n).copy()
    result["Antécédent"] = result["antecedents"].apply(
        lambda values: " + ".join(sorted(values))
    )
    result["Conséquent"] = result["consequents"].apply(
        lambda values: " + ".join(sorted(values))
    )

    return result[
        ["Antécédent", "Conséquent", "support", "confidence", "lift"]
    ].rename(
        columns={
            "support": "Support",
            "confidence": "Confiance",
            "lift": "Lift",
        }
    ).round(3)


# Fonction : plot_rules_sankey
# Input  : DataFrame des règles et nombre de règles affichées.
# Output : diagramme de Sankey Plotly.
def plot_rules_sankey(rules: pd.DataFrame, top_n: int = 15) -> go.Figure:
    """Visualiser les règles d'association sous forme de flux."""
    if rules.empty:
        fig = go.Figure()
        fig.add_annotation(text="Aucune règle disponible", showarrow=False)
        return fig

    top_rules = rules.nlargest(top_n, "lift").copy()
    top_rules["ant_str"] = top_rules["antecedents"].apply(
        lambda values: " + ".join(sorted(values))
    )
    top_rules["cons_str"] = top_rules["consequents"].apply(
        lambda values: " + ".join(sorted(values))
    )

    labels = list(dict.fromkeys(top_rules["ant_str"].tolist() + top_rules["cons_str"].tolist()))
    label_index = {label: index for index, label in enumerate(labels)}

    fig = go.Figure(
        go.Sankey(
            node={"label": labels, "pad": 15, "thickness": 18},
            link={
                "source": [label_index[value] for value in top_rules["ant_str"]],
                "target": [label_index[value] for value in top_rules["cons_str"]],
                "value": top_rules["confidence"].tolist(),
                "customdata": np.stack(
                    [top_rules["support"], top_rules["lift"]], axis=-1
                ),
                "hovertemplate": (
                    "Confiance : %{value:.3f}<br>"
                    "Support : %{customdata[0]:.3f}<br>"
                    "Lift : %{customdata[1]:.3f}<extra></extra>"
                ),
            },
        )
    )
    fig.update_layout(
        title="Principales règles d'association — épaisseur proportionnelle à la confiance",
        height=650,
    )
    return fig

### 6.2 Définition des indicateurs Apriori

- **Support** : part de l'ensemble des incidents contenant simultanément l'antécédent et le conséquent.
- **Confiance** : probabilité d'observer le conséquent lorsque l'antécédent est présent.
- **Lift** : comparaison entre la règle observée et l'indépendance statistique.

\[
\text{Lift}(A \rightarrow B) = \frac{P(B \mid A)}{P(B)}
\]

- un lift supérieur à 1 indique une association positive ;
- un lift proche de 1 indique une association comparable au hasard ;
- un lift inférieur à 1 indique une association négative.

Le seuil de support retenu est **0,10** : une combinaison doit apparaître dans au moins 10 % des incidents, soit environ 95 transactions sur 949. Ce choix limite les coïncidences rares tout en conservant suffisamment de motifs.

**Précaution :** une règle d'association ne prouve pas une causalité. Elle révèle uniquement une cooccurrence statistique dans le dataset.

## 7. Requête 3 — Analyse temporelle et prévision

### 7.1 Agrégations temporelles

L'analyse temporelle mesure le nombre d'incidents par mois et par année. Pour la série mensuelle, les mois sans incident enregistré sont explicitement complétés par zéro afin d'obtenir une chronologie régulière, nécessaire à la prévision.

In [5]:
# Fonction : aggregate_monthly_crimes
# Input  : DataFrame contenant une colonne Date.
# Output : série mensuelle complète avec un nombre d'incidents pour chaque mois.
def aggregate_monthly_crimes(df: pd.DataFrame) -> pd.DataFrame:
    """Compter les incidents par mois et compléter les mois absents par zéro."""
    monthly = (
        df.set_index("Date")
        .resample("MS")
        .size()
        .rename("Crime_Count")
        .reset_index()
    )
    return monthly


# Fonction : aggregate_yearly_crimes
# Input  : DataFrame contenant une colonne Date.
# Output : tableau du nombre d'incidents par année.
def aggregate_yearly_crimes(df: pd.DataFrame) -> pd.DataFrame:
    """Compter les incidents par année."""
    return (
        df.groupby(df["Date"].dt.year)
        .size()
        .rename("Crime_Count")
        .reset_index()
        .rename(columns={"Date": "Year"})
    )


# Fonction : plot_monthly_crimes
# Input  : tableau mensuel.
# Output : courbe Plotly avec annotation du mois maximal.
def plot_monthly_crimes(monthly: pd.DataFrame) -> go.Figure:
    """Visualiser l'évolution mensuelle."""
    peak = monthly.loc[monthly["Crime_Count"].idxmax()]
    fig = px.line(
        monthly,
        x="Date",
        y="Crime_Count",
        title="Évolution mensuelle des incidents à Bridgeport",
        labels={"Date": "Mois", "Crime_Count": "Nombre d'incidents"},
    )
    fig.add_annotation(
        x=peak["Date"],
        y=peak["Crime_Count"],
        text=f"Maximum : {int(peak['Crime_Count'])} ({peak['Date']:%m/%Y})",
        showarrow=True,
    )
    return fig


# Fonction : plot_yearly_crimes
# Input  : tableau annuel.
# Output : courbe Plotly avec annotation de l'année maximale.
def plot_yearly_crimes(yearly: pd.DataFrame) -> go.Figure:
    """Visualiser l'évolution annuelle."""
    peak = yearly.loc[yearly["Crime_Count"].idxmax()]
    fig = px.line(
        yearly,
        x="Year",
        y="Crime_Count",
        markers=True,
        title="Évolution annuelle des incidents à Bridgeport",
        labels={"Year": "Année", "Crime_Count": "Nombre d'incidents"},
    )
    fig.add_annotation(
        x=peak["Year"],
        y=peak["Crime_Count"],
        text=f"Maximum : {int(peak['Crime_Count'])} incidents en {int(peak['Year'])}",
        showarrow=True,
    )
    return fig


# Fonction : prepare_forecast_data
# Input  : tableau mensuel.
# Output : DataFrame avec les colonnes ds et y attendues par Prophet.
def prepare_forecast_data(monthly: pd.DataFrame) -> pd.DataFrame:
    """Adapter la série mensuelle au format Prophet."""
    return monthly.rename(columns={"Date": "ds", "Crime_Count": "y"})[["ds", "y"]]


# Fonction : run_forecast
# Input  : données Prophet et nombre de mois futurs.
# Output : modèle, prévisions et nom de la méthode utilisée.
def run_forecast(
    prophet_df: pd.DataFrame,
    periods: int = 12,
) -> tuple[object | None, pd.DataFrame, str]:
    """Exécuter Prophet ou une prévision de secours si nécessaire."""
    if PROPHET_AVAILABLE:
        try:
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
            )
            model.fit(prophet_df)
            future = model.make_future_dataframe(periods=periods, freq="MS")
            forecast = model.predict(future)
            forecast[["yhat", "yhat_lower", "yhat_upper"]] = forecast[
                ["yhat", "yhat_lower", "yhat_upper"]
            ].clip(lower=0)
            return model, forecast, "Prophet"
        except Exception as exc:
            print(f"Prophet a échoué, utilisation de la méthode de secours : {exc}")

    # Méthode de secours : tendance linéaire + effet moyen du mois.
    history = prophet_df.copy().reset_index(drop=True)
    time_index = np.arange(len(history))
    slope, intercept = np.polyfit(time_index, history["y"], 1)
    trend = intercept + slope * time_index
    residuals = history["y"] - trend
    monthly_effect = residuals.groupby(history["ds"].dt.month).mean()

    future_dates = pd.date_range(
        history["ds"].min(),
        periods=len(history) + periods,
        freq="MS",
    )
    future_index = np.arange(len(future_dates))
    predicted = intercept + slope * future_index
    predicted += pd.Series(future_dates.month).map(monthly_effect).fillna(0).to_numpy()

    sigma = float(residuals.std(ddof=1)) if len(residuals) > 1 else 0.0
    forecast = pd.DataFrame(
        {
            "ds": future_dates,
            "yhat": np.clip(predicted, 0, None),
            "yhat_lower": np.clip(predicted - 1.96 * sigma, 0, None),
            "yhat_upper": np.clip(predicted + 1.96 * sigma, 0, None),
        }
    )
    return None, forecast, "Tendance linéaire avec saisonnalité mensuelle"


# Fonction : plot_forecast
# Input  : historique, prévision et nom de la méthode.
# Output : figure Plotly avec estimation centrale et intervalle d'incertitude.
def plot_forecast(
    prophet_df: pd.DataFrame,
    forecast: pd.DataFrame,
    method_name: str,
) -> go.Figure:
    """Visualiser l'historique et la prévision sur douze mois."""
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=prophet_df["ds"],
            y=prophet_df["y"],
            mode="lines",
            name="Données observées",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=forecast["ds"],
            y=forecast["yhat"],
            mode="lines",
            name=f"Prévision — {method_name}",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=pd.concat([forecast["ds"], forecast["ds"].iloc[::-1]]),
            y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"].iloc[::-1]]),
            fill="toself",
            line={"color": "rgba(0,0,0,0)"},
            name="Intervalle d'incertitude",
        )
    )
    fig.update_layout(
        title="Prévision mensuelle des incidents pour les douze prochains mois",
        xaxis_title="Date",
        yaxis_title="Nombre d'incidents",
    )
    return fig


# Fonction : plot_forecast_components
# Input  : modèle éventuellement entraîné et table de prévision.
# Output : figure des composantes Prophet ou figure de tendance de remplacement.
def plot_forecast_components(model, forecast: pd.DataFrame):
    """Afficher la tendance et la saisonnalité du modèle."""
    if model is not None:
        return model.plot_components(forecast)

    fig = px.line(
        forecast,
        x="ds",
        y="yhat",
        title="Composante principale de la prévision de secours",
        labels={"ds": "Date", "yhat": "Valeur prévue"},
    )
    return fig

### 7.2 Interprétation des indicateurs temporels

- **Nombre mensuel d'incidents** : somme des lignes pour chaque mois. Il montre les fluctuations de court terme.
- **Nombre annuel d'incidents** : somme des lignes pour chaque année. Il permet d'observer la tendance générale.
- **`yhat`** : valeur centrale prévue par le modèle.
- **`yhat_lower` et `yhat_upper`** : bornes de l'intervalle d'incertitude. Plus l'intervalle est large, plus la prévision est incertaine.

L'année 2026 est incomplète dans le fichier, puisque la dernière date disponible est le 1er juin 2026. Son total annuel ne doit donc pas être comparé directement aux années complètes.

Une prévision est une estimation fondée sur les tendances passées. Elle ne constitue pas une certitude et peut être affectée par des changements de politique publique, de collecte des données ou de contexte local.

## 8. Requête 4 — Analyse spatiale et clustering

### 8.1 Préparation géographique

Les lignes sans latitude ou longitude sont retirées uniquement pour l'analyse spatiale. Les coordonnées sont ensuite filtrées dans une emprise cohérente avec Chicago afin d'écarter d'éventuelles valeurs aberrantes.

Un **GeoDataFrame** est également créé avec le système de coordonnées `EPSG:4326`.

In [6]:
# Fonction : prepare_spatial_data
# Input  : DataFrame complet.
# Output : DataFrame contenant uniquement les observations géolocalisables et plausibles.
def prepare_spatial_data(df: pd.DataFrame) -> pd.DataFrame:
    """Nettoyer les coordonnées géographiques."""
    spatial = df.dropna(subset=["Latitude", "Longitude"]).copy()
    spatial = spatial[
        spatial["Latitude"].between(41.6, 42.1)
        & spatial["Longitude"].between(-87.9, -87.5)
    ].copy()
    return spatial


# Fonction : create_geodataframe
# Input  : DataFrame avec Latitude et Longitude.
# Output : GeoDataFrame avec objets Point et projection EPSG:4326.
def create_geodataframe(spatial_df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Créer une géométrie Point pour chaque incident."""
    geometry = [
        Point(longitude, latitude)
        for longitude, latitude in zip(
            spatial_df["Longitude"], spatial_df["Latitude"]
        )
    ]
    return gpd.GeoDataFrame(spatial_df.copy(), geometry=geometry, crs="EPSG:4326")


# Fonction : plot_density_map
# Input  : DataFrame spatial.
# Output : carte de densité interactive Plotly.
def plot_density_map(spatial_df: pd.DataFrame) -> go.Figure:
    """Représenter les concentrations géographiques d'incidents."""
    fig = px.density_map(
        spatial_df,
        lat="Latitude",
        lon="Longitude",
        radius=12,
        center={
            "lat": float(spatial_df["Latitude"].mean()),
            "lon": float(spatial_df["Longitude"].mean()),
        },
        zoom=13,
        map_style="carto-positron",
        title="Densité spatiale des incidents à Bridgeport",
        hover_name="Primary Type",
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    return fig


# Fonction : apply_kmeans
# Input  : DataFrame spatial et nombre de clusters k.
# Output : DataFrame enrichi du cluster K-means et modèle entraîné.
def apply_kmeans(
    spatial_df: pd.DataFrame,
    k: int = 6,
) -> tuple[pd.DataFrame, KMeans]:
    """Regrouper les incidents autour de k centres géographiques."""
    coords = spatial_df[["Latitude", "Longitude"]].to_numpy()
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    result = spatial_df.copy()
    result["cluster_kmeans"] = model.fit_predict(coords).astype(str)
    return result, model


# Fonction : plot_kmeans_map
# Input  : DataFrame enrichi du cluster K-means.
# Output : carte interactive colorée par cluster.
def plot_kmeans_map(kmeans_df: pd.DataFrame) -> go.Figure:
    """Visualiser les groupes géographiques imposés par K-means."""
    fig = px.scatter_map(
        kmeans_df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_kmeans",
        hover_data=["Primary Type", "Location Description", "Arrest"],
        center={
            "lat": float(kmeans_df["Latitude"].mean()),
            "lon": float(kmeans_df["Longitude"].mean()),
        },
        zoom=13,
        map_style="carto-positron",
        title="K-means — six groupes géographiques exploratoires",
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    return fig


# Fonction : apply_optics
# Input  : DataFrame spatial, nombre minimal de voisins et sensibilité xi.
# Output : DataFrame enrichi du cluster OPTICS et modèle entraîné.
def apply_optics(
    spatial_df: pd.DataFrame,
    min_samples: int = 20,
    xi: float = 0.05,
) -> tuple[pd.DataFrame, OPTICS]:
    """Détecter automatiquement les zones denses et le bruit spatial."""
    coords = spatial_df[["Latitude", "Longitude"]].to_numpy()
    model = OPTICS(
        min_samples=min_samples,
        xi=xi,
        min_cluster_size=0.05,
    )
    result = spatial_df.copy()
    result["cluster_optics"] = model.fit_predict(coords).astype(str)
    return result, model


# Fonction : plot_optics_map
# Input  : DataFrame enrichi du cluster OPTICS.
# Output : carte interactive des zones denses et du bruit.
def plot_optics_map(optics_df: pd.DataFrame) -> go.Figure:
    """Visualiser les clusters OPTICS ; la classe -1 correspond au bruit."""
    fig = px.scatter_map(
        optics_df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_optics",
        hover_data=["Primary Type", "Location Description", "Arrest"],
        center={
            "lat": float(optics_df["Latitude"].mean()),
            "lon": float(optics_df["Longitude"].mean()),
        },
        zoom=13,
        map_style="carto-positron",
        title="OPTICS — zones denses et incidents isolés (-1)",
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    return fig


# Fonction : compute_crime_types_by_cluster
# Input  : DataFrame avec clusters K-means.
# Output : tableau des types de crimes par cluster.
def compute_crime_types_by_cluster(kmeans_df: pd.DataFrame) -> pd.DataFrame:
    """Compter les catégories de crimes dans chaque cluster K-means."""
    return (
        kmeans_df.groupby(["cluster_kmeans", "Primary Type"], observed=True)
        .size()
        .reset_index(name="Nombre_incidents")
    )


# Fonction : plot_crime_types_by_cluster
# Input  : tableau produit par compute_crime_types_by_cluster.
# Output : graphique en barres empilées.
def plot_crime_types_by_cluster(cluster_counts: pd.DataFrame) -> go.Figure:
    """Comparer la composition criminelle des clusters K-means."""
    fig = px.bar(
        cluster_counts,
        x="cluster_kmeans",
        y="Nombre_incidents",
        color="Primary Type",
        barmode="stack",
        title="Composition des types de crimes par cluster K-means",
        labels={
            "cluster_kmeans": "Cluster",
            "Nombre_incidents": "Nombre d'incidents",
            "Primary Type": "Type de crime",
        },
    )
    return fig

### 8.2 Interprétation des indicateurs spatiaux

#### Carte de densité

La carte de densité agrège visuellement les incidents proches. Une zone plus intense correspond à une concentration plus importante de points dans le fichier.

#### K-means

K-means impose un nombre de groupes `k`. Chaque incident est rattaché au centre le plus proche. Le choix `k = 6` est conservé pour rester cohérent avec le notebook spatial initial, mais il s'agit d'un **paramètre exploratoire**, et non d'une preuve que Bridgeport se divise naturellement en six zones.

#### OPTICS

OPTICS recherche des zones denses sans imposer directement leur nombre. La catégorie `-1` correspond aux observations considérées comme isolées ou comme du bruit.

**Limites :** les calculs utilisent directement latitude et longitude. Sur un périmètre local restreint, cette approximation reste exploitable pour une exploration, mais une étude opérationnelle devrait projeter les coordonnées dans un système métrique et tester la stabilité des paramètres.

## 9. Intégration du dashboard

In [7]:
# Fonction : launch_dashboard
# Input  : racine du projet et booléen indiquant s'il faut démarrer le serveur.
# Output : chemin du fichier app.py ou processus serveur lorsque start_server=True.
def launch_dashboard(project_root: Path, start_server: bool = False):
    """Vérifier la présence du dashboard et éventuellement démarrer Dash."""
    app_path = project_root / "dashboard" / "app.py"

    if not app_path.exists():
        print(
            "Dashboard non trouvé. Ajoutez dashboard/app.py avant la remise finale."
        )
        return None

    if start_server:
        # Cette commande bloque la cellule tant que le serveur Dash fonctionne.
        # Utiliser Ctrl+C dans le terminal ou interrompre le kernel pour l'arrêter.
        return subprocess.run([sys.executable, str(app_path)], check=True)

    print(f"Dashboard détecté : {app_path}")
    print("Commande de lancement : python dashboard/app.py")
    return app_path

Le dashboard est un livrable distinct du notebook. Il organise les résultats des quatre requêtes et doit être lancé depuis la racine du projet avec :

```bash
python dashboard/app.py
```

Pour éviter de bloquer l'exécution automatique du notebook, le serveur n'est pas démarré par défaut dans le bloc principal. Pendant la démonstration, il suffit d'utiliser la commande ci-dessus ou de passer `start_dashboard=True` dans `main()`.

## 10. Bloc principal — exécution de toutes les fonctions

In [8]:
# Fonction : main
# Input  : chemin du CSV et option de lancement du serveur Dash.
# Output : dictionnaire contenant toutes les tables, figures et modèles produits.
def main(data_path: str | Path, start_dashboard: bool = False) -> dict:
    """Exécuter l'ensemble du processus KDD dans l'ordre logique."""
    project_root = resolve_project_root()

    print("=" * 80)
    print("CHARGEMENT ET EXPLORATION")
    print("=" * 80)
    df = load_data(data_path)
    overview = compute_dataset_overview(df)
    variable_summary = build_variable_summary(df)
    top_crimes = compute_top_crimes(df, top_n=10)
    arrest_rates = compute_arrest_rates(df, top_n=10)
    fig_top_crimes = plot_top_crimes(top_crimes)
    fig_arrest_rates = plot_arrest_rates(arrest_rates)

    display(pd.DataFrame([overview]).T.rename(columns={0: "Valeur"}))
    display(variable_summary)
    display(top_crimes)
    display(fig_top_crimes)
    display(arrest_rates)
    display(fig_arrest_rates)

    print("=" * 80)
    print("PATTERN MINING")
    print("=" * 80)
    discrete_df = discretise_for_apriori(df)
    encoded_df = encode_transactions(discrete_df)
    itemsets, rules = extract_frequent_patterns(
        encoded_df,
        min_support=0.10,
        min_lift=1.0,
    )
    sensitivity = compute_support_sensitivity(encoded_df)
    rules_table = format_rules_table(rules, top_n=15)
    fig_support = plot_support_sensitivity(sensitivity)
    fig_sankey = plot_rules_sankey(rules, top_n=15)

    print(f"Itemsets fréquents : {len(itemsets)}")
    print(f"Règles d'association : {len(rules)}")
    display(discrete_df.head())
    display(sensitivity)
    display(fig_support)
    display(rules_table)
    display(fig_sankey)

    print("=" * 80)
    print("ANALYSE TEMPORELLE ET PRÉVISION")
    print("=" * 80)
    monthly = aggregate_monthly_crimes(df)
    yearly = aggregate_yearly_crimes(df)
    fig_monthly = plot_monthly_crimes(monthly)
    fig_yearly = plot_yearly_crimes(yearly)
    prophet_df = prepare_forecast_data(monthly)
    forecast_model, forecast, forecast_method = run_forecast(prophet_df, periods=12)
    fig_forecast = plot_forecast(prophet_df, forecast, forecast_method)
    fig_components = plot_forecast_components(forecast_model, forecast)

    monthly_peak = monthly.loc[monthly["Crime_Count"].idxmax()]
    yearly_peak = yearly.loc[yearly["Crime_Count"].idxmax()]
    print(
        f"Mois maximal : {monthly_peak['Date']:%m/%Y} "
        f"avec {int(monthly_peak['Crime_Count'])} incidents"
    )
    print(
        f"Année maximale : {int(yearly_peak['Year'])} "
        f"avec {int(yearly_peak['Crime_Count'])} incidents"
    )
    print(f"Méthode de prévision utilisée : {forecast_method}")
    display(fig_monthly)
    display(yearly)
    display(fig_yearly)
    display(forecast.tail(12)[["ds", "yhat", "yhat_lower", "yhat_upper"]].round(2))
    display(fig_forecast)
    display(fig_components)

    print("=" * 80)
    print("ANALYSE SPATIALE")
    print("=" * 80)
    spatial_df = prepare_spatial_data(df)
    geo_df = create_geodataframe(spatial_df)
    fig_density = plot_density_map(spatial_df)
    kmeans_df, kmeans_model = apply_kmeans(spatial_df, k=6)
    fig_kmeans = plot_kmeans_map(kmeans_df)
    optics_df, optics_model = apply_optics(spatial_df, min_samples=20, xi=0.05)
    fig_optics = plot_optics_map(optics_df)
    cluster_counts = compute_crime_types_by_cluster(kmeans_df)
    fig_cluster_types = plot_crime_types_by_cluster(cluster_counts)

    optics_counts = optics_df["cluster_optics"].value_counts().sort_index()
    print(f"Observations géolocalisées : {len(spatial_df)}")
    print("Effectifs K-means :")
    display(kmeans_df["cluster_kmeans"].value_counts().sort_index().rename("Effectif"))
    print("Effectifs OPTICS (-1 = bruit) :")
    display(optics_counts.rename("Effectif"))
    display(geo_df[["Primary Type", "Latitude", "Longitude", "geometry"]].head())
    display(fig_density)
    display(fig_kmeans)
    display(fig_optics)
    display(fig_cluster_types)

    print("=" * 80)
    print("DASHBOARD")
    print("=" * 80)
    dashboard_path = launch_dashboard(project_root, start_server=start_dashboard)

    return {
        "data": df,
        "overview": overview,
        "variable_summary": variable_summary,
        "top_crimes": top_crimes,
        "arrest_rates": arrest_rates,
        "discrete_data": discrete_df,
        "encoded_data": encoded_df,
        "itemsets": itemsets,
        "rules": rules,
        "support_sensitivity": sensitivity,
        "monthly": monthly,
        "yearly": yearly,
        "forecast_model": forecast_model,
        "forecast": forecast,
        "forecast_method": forecast_method,
        "spatial_data": spatial_df,
        "geodataframe": geo_df,
        "kmeans_model": kmeans_model,
        "optics_model": optics_model,
        "dashboard_path": dashboard_path,
    }


# Configuration d'exécution du notebook.
PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "chicago_crime.csv"

# Laisser False pour une exécution normale du notebook.
# Passer à True uniquement pendant la démonstration si l'on souhaite démarrer Dash ici.
START_DASHBOARD_SERVER = False

results = main(DATA_PATH, start_dashboard=START_DASHBOARD_SERVER)

CHARGEMENT ET EXPLORATION


,Valeur
Nombre de lignes,949
Nombre de colonnes,23
Date minimale,2002-04-21 18:10:00
Date maximale,2026-06-01 21:58:00
Types de crimes distincts,20
Taux global d'arrestation (%),34.77
Lignes sans coordonnées complètes,6


,Variable,Type,Valeurs manquantes,Taux manquant (%),Valeurs distinctes,Minimum / information 1,Maximum / information 2
0,ID,int64,0,0.00,949,2100407,14216537
1,Case Number,str,0,0.00,949,Modalité fréquente : HH322288,949 modalités distinctes
2,Date,datetime64[us],0,0.00,948,2002-04-21 18:10:00,2026-06-01 21:58:00
3,Block,str,0,0.00,1,Modalité fréquente : 007XX W 31ST ST,1 modalités distinctes
4,IUCR,str,0,0.00,91,Modalité fréquente : 0860,91 modalités distinctes
5,Primary Type,str,0,0.00,20,Modalité fréquente : THEFT,20 modalités distinctes
6,Description,str,0,0.00,95,Modalité fréquente : RETAIL THEFT,95 modalités distinctes
7,Location Description,str,0,0.00,37,Modalité fréquente : DRUG STORE,37 modalités distinctes
8,Arrest,bool,0,0.00,2,False,True
9,Domestic,bool,0,0.00,2,False,True


,Primary Type,Nombre_incidents
18,THEFT,387
0,ASSAULT,118
1,BATTERY,103
5,DECEPTIVE PRACTICE,66
4,CRIMINAL TRESPASS,63
3,CRIMINAL DAMAGE,60
12,NARCOTICS,34
14,OTHER OFFENSE,31
2,BURGLARY,28
16,ROBBERY,21


,Primary Type,Nombre_incidents,Nombre_arrestations,Taux_arrestation_pct
6,NARCOTICS,34,33,97.06
4,CRIMINAL TRESPASS,63,55,87.30
0,THEFT,387,133,34.37
1,ASSAULT,118,36,30.51
9,ROBBERY,21,6,28.57
2,BATTERY,103,27,26.21
7,OTHER OFFENSE,31,8,25.81
3,DECEPTIVE PRACTICE,66,5,7.58
5,CRIMINAL DAMAGE,60,3,5.00
8,BURGLARY,28,1,3.57


...

cluster_kmeans
0    571
1     88
2     59
3    179
4     43
5      3
Name: Effectif, dtype: int64

Effectifs OPTICS (-1 = bruit) :


cluster_optics
-1    128
0     480
1      57
2      91
3      99
4      88
Name: Effectif, dtype: int64

,Primary Type,Latitude,Longitude,geometry
0,ASSAULT,41.838055,-87.645458,POINT (-87.64546 41.83805)
1,THEFT,41.838053,-87.645565,POINT (-87.64556 41.83805)
2,THEFT,41.838055,-87.645458,POINT (-87.64546 41.83805)
3,ASSAULT,41.838055,-87.645458,POINT (-87.64546 41.83805)
4,THEFT,41.838053,-87.645565,POINT (-87.64556 41.83805)


DASHBOARD
Dashboard détecté : /mnt/data/chicago_project/chicago-crime-analysis-main/dashboard/app.py
Commande de lancement : python dashboard/app.py


## 11. Synthèse générale des résultats

Les résultats produits par le notebook permettent de retenir les éléments suivants :

1. Le dataset contient **949 incidents** et **20 variables d'origine**, sur une période allant du 21 avril 2002 au 1er juin 2026.
2. Le type de crime le plus représenté est `THEFT`.
3. Le taux global d'arrestation est d'environ 35 %, mais il varie fortement selon la catégorie d'infraction.
4. Avec un support minimal de 0,10, Apriori met en évidence des associations récurrentes entre lieu, arrestation, caractère domestique, tranche horaire et type de crime.
5. Le maximum mensuel observé se situe en juin 2005, tandis que le maximum annuel se situe en 2011.
6. Les cartes montrent que les incidents ne sont pas répartis uniformément dans le périmètre étudié.
7. K-means fournit une partition imposée en six groupes, tandis qu'OPTICS détecte automatiquement des zones denses et des observations isolées.

Ces résultats sont descriptifs et exploratoires. Ils peuvent soutenir une réflexion ou une prise de décision, mais ils ne permettent pas d'établir à eux seuls des relations causales.

## 12. Limites et précautions d'interprétation

- Le dataset est limité au secteur de Bridgeport et n'est pas représentatif de tout Chicago.
- Les incidents enregistrés dépendent des pratiques de signalement, de saisie et de mise à jour des services de police.
- L'année 2026 est incomplète.
- Les règles Apriori décrivent des associations et non des causes.
- Une forte confiance peut correspondre à une conséquence très fréquente ; le lift doit donc être examiné conjointement.
- Les paramètres de K-means et d'OPTICS influencent les clusters obtenus.
- Les prévisions reposent sur l'historique disponible et restent incertaines.
- Aucun dénominateur de population ou de fréquentation n'est disponible : les résultats sont des nombres bruts, et non des taux de criminalité par habitant ou par usager.

## 13. Commandes finales pour la démonstration

Depuis la racine du dépôt :

```bash
python -m pip install -r dashboard/requirements.txt
jupyter notebook notebooks/Kavuansiko_notebook.ipynb
```

Dans un second terminal, pour lancer le dashboard :

```bash
python dashboard/app.py
```

Le navigateur doit ensuite ouvrir :

```text
http://127.0.0.1:8050
```

Avant le dépôt, remplacer impérativement la mention `À COMPLÉTER AVANT LE DÉPÔT` par l'adresse publique du dépôt GitHub.